In [ ]:
"""
Paddy Leaf Disease Classification Using EfficientNet-B0
Complete implementation with visualization and performance metrics
Target Accuracy: 98%+
"""

# ==================== IMPORTS ====================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, confusion_matrix, classification_report,
                            roc_curve, auc, precision_recall_curve)
from sklearn.preprocessing import label_binarize
from itertools import cycle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow.keras import backend as K

K.clear_session()
tf.keras.backend.clear_session()


In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ==================== CONFIGURATION ====================
# Plot configuration
plt.rcParams["figure.figsize"] = (11, 7)
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 18
plt.rcParams['font.weight'] = 'bold'

# Dataset configuration
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 3
CLASS_NAMES = ['Bacterial Leaf Blight', 'Brown Spot', 'Leaf Smut']

# Dataset path (modify according to your local path)
TRAIN_PATH = './dataset/train'
VALID_PATH = './dataset/valid'
TEST_PATH  = './dataset/test'


# ==================== DATA PREPARATION ====================
print("=" * 80)
print("STEP 1: DATA PREPARATION")
print("=" * 80)

# Data augmentation for training (to overcome small dataset size)

# ==================== DATA GENERATORS ====================

# Training data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Validation & Test (NO augmentation)
val_test_datagen = ImageDataGenerator(rescale=1./255)


train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='rgb', 
    shuffle=True,
    seed=42
)

validation_generator = val_test_datagen.flow_from_directory(
    VALID_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
      color_mode='rgb', 
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
      color_mode='rgb', 
    shuffle=False
)

print(f"Training samples   : {train_generator.samples}")
print(f"Validation samples : {validation_generator.samples}")
print(f"Test samples       : {test_generator.samples}")
print(f"Class indices      : {train_generator.class_indices}")
print(f"Image shape        : {train_generator.image_shape}")


In [ ]:
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2B0
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

def build_model(num_classes=NUM_CLASSES, img_size=IMG_SIZE):

    base_model = EfficientNetV2B0(
        include_top=False,
        weights="imagenet",
        input_shape=(img_size, img_size, 3)
    )

    base_model.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(base_model.input, outputs, name="EfficientNetV2B0")

    return model, base_model




model, base_model = build_efficientnet_model(3, IMG_SIZE)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()
print(model.input_shape)





In [ ]:
"""
Paddy Leaf Disease Classification Using EfficientNet-B0
Complete implementation with visualization and performance metrics
Target Accuracy: 98%+
"""

# ==================== IMPORTS ====================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, confusion_matrix, classification_report,
                            roc_curve, auc, precision_recall_curve)
from sklearn.preprocessing import label_binarize
from itertools import cycle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ==================== CONFIGURATION ====================
# Plot configuration
plt.rcParams["figure.figsize"] = (11, 7)
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 18
plt.rcParams['font.weight'] = 'bold'

# Dataset configuration
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 3
CLASS_NAMES = ['Bacterial Leaf Blight', 'Brown Spot', 'Leaf Smut']

# Dataset path (modify according to your local path)
DATASET_PATH = './dataset/train'  # Change this to your dataset path

# ==================== DATA PREPARATION ====================
print("=" * 80)
print("STEP 1: DATA PREPARATION")
print("=" * 80)

# Data augmentation for training (to overcome small dataset size)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.15  # 15% for validation
)

# Only rescaling for validation and test (no augmentation)
test_datagen = ImageDataGenerator(
    rescale=1./255
)

# Load training data (70% of total data)
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

# Load validation data (15% of total data)
validation_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

# For testing, you should have a separate test folder with 15% of data
# If you don't have a separate test folder, we'll use validation as test
test_generator = validation_generator

print(f"\nTraining samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Test samples: {test_generator.samples}")
print(f"Class indices: {train_generator.class_indices}")
print(f"Training generator shape: {train_generator.image_shape}")

# ==================== MODEL BUILDING ====================
print("\n" + "=" * 80)
print("STEP 2: MODEL BUILDING - EfficientNet-B0 with Transfer Learning")
print("=" * 80)

def build_efficientnet_model(num_classes=NUM_CLASSES, img_size=IMG_SIZE):
    """
    Build EfficientNet-B0 model with transfer learning
    """
    # Load pre-trained EfficientNet-B0 (without top layer)
    # Create input layer explicitly
    input_layer = layers.Input(shape=(img_size, img_size, 3), name='input_image')

    base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_tensor=input_layer,  # ✅ Use input_tensor instead
    pooling=None
    )
    # Freeze base model layers initially
    base_model.trainable = False
    
    # Build the model
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

# Build model
model, base_model = build_efficientnet_model(3,IMG_SIZE)

# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel Summary:")
model.summary()

# ==================== CALLBACKS ====================
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# ==================== TRAINING PHASE 1 ====================
print("\n" + "=" * 80)
print("STEP 3: TRAINING PHASE 1 - Frozen Base Model")
print("=" * 80)

history_phase1 = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

# ==================== FINE-TUNING PHASE 2 ====================
print("\n" + "=" * 80)
print("STEP 4: FINE-TUNING PHASE 2 - Unfrozen Base Model")
print("=" * 80)

# Unfreeze the base model for fine-tuning
base_model.trainable = True

# Fine-tune from layer 100 onwards
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_phase2 = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=30,
    initial_epoch=20,
    callbacks=callbacks,
    verbose=1
)

# ==================== COMBINE TRAINING HISTORY ====================
# Combine both training phases
history = {
    'accuracy': history_phase1.history['accuracy'] + history_phase2.history['accuracy'],
    'val_accuracy': history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy'],
    'loss': history_phase1.history['loss'] + history_phase2.history['loss'],
    'val_loss': history_phase1.history['val_loss'] + history_phase2.history['val_loss']
}

# ==================== EVALUATION ====================
print("\n" + "=" * 80)
print("STEP 5: MODEL EVALUATION")
print("=" * 80)

# Get predictions
test_generator.reset()
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\n{'='*80}")
print("PERFORMANCE METRICS")
print(f"{'='*80}")
print(f"Accuracy:  {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1-Score:  {f1*100:.2f}%")
print(f"{'='*80}")

# Detailed classification report
print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# ==================== VISUALIZATION ====================
print("\n" + "=" * 80)
print("STEP 6: GENERATING VISUALIZATIONS")
print("=" * 80)

# Create output directory
os.makedirs('outputs', exist_ok=True)

# 1. ACCURACY CURVE
plt.figure(figsize=(11, 7), dpi=800)
epochs_range = range(1, len(history['accuracy']) + 1)
plt.plot(epochs_range, history['accuracy'], 'b-', linewidth=2.5, label='Training Accuracy')
plt.plot(epochs_range, history['val_accuracy'], 'r-', linewidth=2.5, label='Validation Accuracy')
plt.axvline(x=20, color='green', linestyle='--', linewidth=2, label='Fine-tuning Start')
plt.xlabel('Epochs', fontweight='bold')
plt.ylabel('Accuracy', fontweight='bold')
plt.title('Model Accuracy Curve', fontweight='bold', pad=20)
plt.legend(loc='lower right', frameon=True, shadow=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/accuracy_curve.png', dpi=800, bbox_inches='tight')
plt.close()
print("✓ Accuracy curve saved")

# 2. LOSS CURVE
plt.figure(figsize=(11, 7), dpi=800)
plt.plot(epochs_range, history['loss'], 'b-', linewidth=2.5, label='Training Loss')
plt.plot(epochs_range, history['val_loss'], 'r-', linewidth=2.5, label='Validation Loss')
plt.axvline(x=20, color='green', linestyle='--', linewidth=2, label='Fine-tuning Start')
plt.xlabel('Epochs', fontweight='bold')
plt.ylabel('Loss', fontweight='bold')
plt.title('Model Loss Curve', fontweight='bold', pad=20)
plt.legend(loc='upper right', frameon=True, shadow=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/loss_curve.png', dpi=800, bbox_inches='tight')
plt.close()
print("✓ Loss curve saved")

# 3. CONFUSION MATRIX
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(11, 7), dpi=800)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cbar_kws={'label': 'Count'}, linewidths=2, linecolor='black',
            annot_kws={'size': 16, 'weight': 'bold'})
plt.xlabel('Predicted Label', fontweight='bold')
plt.ylabel('True Label', fontweight='bold')
plt.title('Confusion Matrix', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=800, bbox_inches='tight')
plt.close()
print("✓ Confusion matrix saved")

# 4. ROC CURVE (Multi-class)
y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
fpr = dict()
tpr = dict()
roc_auc = dict()

plt.figure(figsize=(11, 7), dpi=800)
colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(NUM_CLASSES), colors):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], color=color, linewidth=2.5,
             label=f'{CLASS_NAMES[i]} (AUC = {roc_auc[i]:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontweight='bold')
plt.ylabel('True Positive Rate', fontweight='bold')
plt.title('ROC Curve - Multi-class', fontweight='bold', pad=20)
plt.legend(loc='lower right', frameon=True, shadow=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/roc_curve.png', dpi=800, bbox_inches='tight')
plt.close()
print("✓ ROC curve saved")

# 5. PRECISION-RECALL CURVE
plt.figure(figsize=(11, 7), dpi=800)
colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(NUM_CLASSES), colors):
    precision_vals, recall_vals, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
    pr_auc = auc(recall_vals, precision_vals)
    plt.plot(recall_vals, precision_vals, color=color, linewidth=2.5,
             label=f'{CLASS_NAMES[i]} (AUC = {pr_auc:.3f})')

plt.xlabel('Recall', fontweight='bold')
plt.ylabel('Precision', fontweight='bold')
plt.title('Precision-Recall Curve', fontweight='bold', pad=20)
plt.legend(loc='lower left', frameon=True, shadow=True)
plt.grid(True, alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.tight_layout()
plt.savefig('outputs/precision_recall_curve.png', dpi=800, bbox_inches='tight')
plt.close()
print("✓ Precision-Recall curve saved")

# 6. PERFORMANCE METRICS BAR CHART
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrics_values = [accuracy * 100, precision * 100, recall * 100, f1 * 100]

plt.figure(figsize=(11, 7), dpi=800)
bars = plt.bar(metrics_names, metrics_values, color=['#2E86AB', '#A23B72', '#F18F01', '#06A77D'],
               edgecolor='black', linewidth=2, width=0.6)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}%',
             ha='center', va='bottom', fontweight='bold', fontsize=16)

plt.ylabel('Score (%)', fontweight='bold')
plt.title('Performance Metrics', fontweight='bold', pad=20)
plt.ylim([0, 105])
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('outputs/performance_metrics.png', dpi=800, bbox_inches='tight')
plt.close()
print("✓ Performance metrics chart saved")

# ==================== SAVE RESULTS ====================
# Save metrics to CSV
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Value (%)': [accuracy * 100, precision * 100, recall * 100, f1 * 100]
})
metrics_df.to_csv('outputs/performance_metrics.csv', index=False)

# Save confusion matrix to CSV
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.to_csv('outputs/confusion_matrix.csv')

# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv('outputs/training_history.csv', index=False)

print("\n" + "=" * 80)
print("TRAINING AND EVALUATION COMPLETED SUCCESSFULLY!")
print("=" * 80)
print("\nAll outputs saved in 'outputs/' directory:")
print("  - accuracy_curve.png")
print("  - loss_curve.png")
print("  - confusion_matrix.png")
print("  - roc_curve.png")
print("  - precision_recall_curve.png")
print("  - performance_metrics.png")
print("  - performance_metrics.csv")
print("  - confusion_matrix.csv")
print("  - training_history.csv")
print("  - best_model.h5")
print("=" * 80)

# Save final model
model.save('outputs/final_model.h5')
print("\n✓ Final model saved as 'outputs/final_model.h5'")
print("\n" + "=" * 80)
print(f"FINAL ACCURACY: {accuracy*100:.2f}%")
print("=" * 80)